In [0]:
dbutils.widgets.text("catalog_name",  "dbw_chinook_team", "Catalog")
dbutils.widgets.text("silver_schema", "chinook_silver",   "Silver")
dbutils.widgets.text("gold_schema",   "chinook_gold",     "Gold")

In [0]:
from pyspark.sql.functions import (
    col, concat_ws, md5, lit, to_date,
    current_timestamp, date_format,
    coalesce, sum, count, avg, max, min
)
from delta.tables import DeltaTable

CATALOG = dbutils.widgets.get("catalog_name")
SILVER  = f"{CATALOG}.{dbutils.widgets.get('silver_schema')}"
GOLD    = f"{CATALOG}.{dbutils.widgets.get('gold_schema')}"

print(f"Catalog : {CATALOG}")
print(f"Silver  : {SILVER}")
print(f"Gold    : {GOLD}")

In [0]:
# ── DIM_DATE ─────────────────────────────────────────────────────────────────
# Static date spine — safe to overwrite on every run
print("Building DIM_DATE...")
date_dim = spark.sql("""
    SELECT
        CAST(date_format(d,'yyyyMMdd') AS INT) AS date_sk,
        d                                       AS full_date,
        year(d)                                 AS year,
        quarter(d)                              AS quarter,
        month(d)                                AS month,
        date_format(d,'MMMM')                   AS month_name,
        dayofmonth(d)                           AS day,
        dayofweek(d)                            AS day_of_week,
        date_format(d,'EEEE')                   AS day_name,
        weekofyear(d)                           AS week_of_year
    FROM (
        SELECT explode(sequence(
            to_date('2000-01-01'),
            to_date('2030-12-31'),
            interval 1 day
        )) AS d
    )
""")
date_dim.write.format("delta").mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable(f"{GOLD}.dim_date")
print(f"✅ DIM_DATE: {date_dim.count()} rows")

In [0]:
# ── DIM_CUSTOMER (SCD Type 2) ─────────────────────────────────────────────────
#
# First run  : writes all rows as is_current=True, effective_end_date=NULL
# Subsequent : compares hash_value to detect changes, then:
#   1. Expires old rows  (is_current=False, effective_end_date=now)
#   2. Appends new rows  (new customers + updated versions of changed ones)
#
# hash_value covers all slowly-changing attributes so unchanged customers
# are never touched.
# ─────────────────────────────────────────────────────────────────────────────
print("Building DIM_CUSTOMER (SCD Type 2)...")

customer_silver = spark.table(f"{SILVER}.customer")

incoming = customer_silver.select(
    col("CustomerId").alias("customer_id"),
    concat_ws(" ", col("FirstName"), col("LastName")).alias("customer_name"),
    col("FirstName").alias("first_name"),
    col("LastName").alias("last_name"),
    col("Email").alias("email"),
    col("Phone").alias("phone"),
    col("Address").alias("address"),
    col("City").alias("city"),
    col("State").alias("state"),
    col("Country").alias("country"),
    col("PostalCode").alias("postal_code"),
    col("Company").alias("company"),
    col("SupportRepId").alias("support_rep_id")
).withColumn("hash_value",
    md5(concat_ws("||",
        coalesce(col("first_name"),  lit("")),
        coalesce(col("last_name"),   lit("")),
        coalesce(col("email"),       lit("")),
        coalesce(col("phone"),       lit("")),
        coalesce(col("address"),     lit("")),
        coalesce(col("city"),        lit("")),
        coalesce(col("state"),       lit("")),
        coalesce(col("country"),     lit("")),
        coalesce(col("postal_code"), lit(""))
    ))
).withColumn("customer_key",
    md5(col("customer_id").cast("string"))) \
 .withColumn("effective_start_date", current_timestamp()) \
 .withColumn("effective_end_date",   lit(None).cast("timestamp")) \
 .withColumn("is_current",           lit(True))

# ── FIRST RUN: table does not exist yet ──────────────────────────────────────
if not spark.catalog.tableExists(f"{GOLD}.dim_customer"):
    incoming.write.format("delta") \
        .mode("overwrite") \
        .option("overwriteSchema", "true") \
        .saveAsTable(f"{GOLD}.dim_customer")
    print(f"✅ DIM_CUSTOMER (initial load): {incoming.count()} rows")

# ── SUBSEQUENT RUNS: SCD Type 2 MERGE ────────────────────────────────────────
else:
    target      = DeltaTable.forName(spark, f"{GOLD}.dim_customer")
    target_curr = target.toDF().filter("is_current = true")

    # Step 1 — detect changed customers (hash mismatch against current row)
    changed = (
        incoming.alias("src")
        .join(target_curr.alias("tgt"), on="customer_key")
        .filter("src.hash_value != tgt.hash_value")
        .select(col("src.customer_key"))
    )
    changed_keys = [r.customer_key for r in changed.collect()]
    changed_count = len(changed_keys)
    print(f"  Changed customers detected : {changed_count}")

    # Step 2 — expire old current rows for changed customers
    if changed_count > 0:
        keys_literal = ", ".join(f"'{k}'" for k in changed_keys)
        target.update(
            condition=f"is_current = true AND customer_key IN ({keys_literal})",
            set={
                "is_current":         lit(False),
                "effective_end_date": current_timestamp()
            }
        )
        print(f"  Expired {changed_count} old row(s)")

    # Step 3 — identify brand-new customers (no existing row at all)
    new_only = (
        incoming.alias("src")
        .join(target_curr.alias("tgt"), on="customer_key", how="left_anti")
    )

    # Step 4 — build insert set: new customers + updated versions of changed ones
    if changed_count > 0:
        changed_df  = spark.createDataFrame([(k,) for k in changed_keys], ["customer_key"])
        updated_rows = incoming.join(changed_df, on="customer_key")
        to_insert    = new_only.union(updated_rows)
    else:
        to_insert = new_only

    insert_count = to_insert.count()
    if insert_count > 0:
        to_insert.write.format("delta") \
            .mode("append") \
            .saveAsTable(f"{GOLD}.dim_customer")
        print(f"  Inserted {insert_count} new/updated row(s)")
    else:
        print("  No new or changed customers — nothing to insert")

    print(f"✅ DIM_CUSTOMER (SCD2 merge complete): {spark.table(f'{GOLD}.dim_customer').count()} total rows")   

In [0]:
# ── DIM_TRACK ─────────────────────────────────────────────────────────────────
# Denormalized join of track + album + artist + genre + mediatype.
# Chinook tracks don't change — safe to overwrite on every run.
# ─────────────────────────────────────────────────────────────────────────────
print("Building DIM_TRACK...")
track     = spark.table(f"{SILVER}.track")
album     = spark.table(f"{SILVER}.album")
artist    = spark.table(f"{SILVER}.artist")
genre     = spark.table(f"{SILVER}.genre")
mediatype = spark.table(f"{SILVER}.mediatype")

track_dim = track \
    .join(album,     "AlbumId",     "left") \
    .join(artist,    "ArtistId",    "left") \
    .join(genre,     "GenreId",     "left") \
    .join(mediatype, "MediaTypeId", "left") \
    .select(
        track["TrackId"].alias("track_nk"),
        md5(track["TrackId"].cast("string")).alias("track_key"),
        track["Name"].alias("track_name"),
        album["Title"].alias("album_title"),
        artist["Name"].alias("artist_name"),
        genre["Name"].alias("genre_name"),
        mediatype["Name"].alias("media_type_name"),
        track["Composer"].alias("composer"),
        track["Milliseconds"].alias("duration_ms"),
        track["UnitPrice"].cast("decimal(10,2)").alias("unit_price")
    )

track_dim.write.format("delta").mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable(f"{GOLD}.dim_track")
print(f"✅ DIM_TRACK: {track_dim.count()} rows")

In [0]:
# ── DIM_EMPLOYEE (SCD Type 2) ─────────────────────────────────────────────────
print("Building DIM_EMPLOYEE (SCD Type 2)...")

employee_silver = spark.table(f"{SILVER}.employee")

incoming_emp = employee_silver.select(
    col("EmployeeId").alias("employee_nk"),
    md5(col("EmployeeId").cast("string")).alias("employee_key"),
    concat_ws(" ", col("FirstName"), col("LastName")).alias("employee_name"),
    col("Title").alias("title"),
    col("ReportsTo").alias("reports_to_nk"),
    col("BirthDate").alias("birth_date"),
    col("HireDate").alias("hire_date"),
    col("City").alias("city"),
    col("State").alias("state"),
    col("Country").alias("country"),
    col("Email").alias("email")
).withColumn("hash_value",
    md5(concat_ws("||",
        coalesce(col("title"),         lit("")),
        coalesce(col("city"),          lit("")),
        coalesce(col("state"),         lit("")),
        coalesce(col("country"),       lit("")),
        coalesce(col("email"),         lit("")),
        coalesce(col("reports_to_nk").cast("string"), lit(""))
    ))
) \
 .withColumn("effective_start_date", current_timestamp()) \
 .withColumn("effective_end_date",   lit(None).cast("timestamp")) \
 .withColumn("is_current",           lit(True))

# Check if table exists AND has the SCD2 columns already
table_exists = spark.catalog.tableExists(f"{GOLD}.dim_employee")
is_scd2 = table_exists and "is_current" in [f.name for f in spark.table(f"{GOLD}.dim_employee").schema.fields]

# ── FIRST RUN (or schema migration from old non-SCD2 table) ──────────────────
if not is_scd2:
    incoming_emp.write.format("delta") \
        .mode("overwrite") \
        .option("overwriteSchema", "true") \
        .saveAsTable(f"{GOLD}.dim_employee")
    print(f"✅ DIM_EMPLOYEE (initial load): {incoming_emp.count()} rows")

# ── SUBSEQUENT RUNS: SCD Type 2 MERGE ────────────────────────────────────────
else:
    target_emp      = DeltaTable.forName(spark, f"{GOLD}.dim_employee")
    target_emp_curr = target_emp.toDF().filter("is_current = true")

    changed_emp = (
        incoming_emp.alias("src")
        .join(target_emp_curr.alias("tgt"), on="employee_key")
        .filter("src.hash_value != tgt.hash_value")
        .select(col("src.employee_key"))
    )
    changed_emp_keys  = [r.employee_key for r in changed_emp.collect()]
    changed_emp_count = len(changed_emp_keys)
    print(f"  Changed employees detected : {changed_emp_count}")

    if changed_emp_count > 0:
        emp_keys_literal = ", ".join(f"'{k}'" for k in changed_emp_keys)
        target_emp.update(
            condition=f"is_current = true AND employee_key IN ({emp_keys_literal})",
            set={
                "is_current":         lit(False),
                "effective_end_date": current_timestamp()
            }
        )
        print(f"  Expired {changed_emp_count} old row(s)")

    new_emp_only = (
        incoming_emp.alias("src")
        .join(target_emp_curr.alias("tgt"), on="employee_key", how="left_anti")
    )

    if changed_emp_count > 0:
        changed_emp_df   = spark.createDataFrame([(k,) for k in changed_emp_keys], ["employee_key"])
        updated_emp_rows = incoming_emp.join(changed_emp_df, on="employee_key")
        to_insert_emp    = new_emp_only.union(updated_emp_rows)
    else:
        to_insert_emp = new_emp_only

    insert_emp_count = to_insert_emp.count()
    if insert_emp_count > 0:
        to_insert_emp.write.format("delta") \
            .mode("append") \
            .saveAsTable(f"{GOLD}.dim_employee")
        print(f"  Inserted {insert_emp_count} new/updated row(s)")
    else:
        print("  No new or changed employees — nothing to insert")

    print(f"✅ DIM_EMPLOYEE (SCD2 merge complete): {spark.table(f'{GOLD}.dim_employee').count()} total rows")

In [0]:
# FACT_SALES
print("Building FACT_SALES...")
invoice      = spark.table(f"{SILVER}.invoice")
invoiceline  = spark.table(f"{SILVER}.invoiceline")
dim_customer = spark.table(f"{GOLD}.dim_customer").filter("is_current = true")
dim_track    = spark.table(f"{GOLD}.dim_track")
dim_employee = spark.table(f"{GOLD}.dim_employee")
dim_date     = spark.table(f"{GOLD}.dim_date")
customer_sv  = spark.table(f"{SILVER}.customer")

invoice_keyed = invoice.withColumn(
    "date_sk", date_format(col("InvoiceDate"),"yyyyMMdd").cast("int")
)

sales_fact = invoiceline \
    .join(invoice_keyed, "InvoiceId", "inner") \
    .join(
        dim_customer.select("customer_id","customer_key"),
        invoice_keyed["CustomerId"] == dim_customer["customer_id"], "left"
    ) \
    .join(
        dim_track.select("track_nk","track_key"),
        invoiceline["TrackId"] == dim_track["track_nk"], "left"
    ) \
    .join(
        dim_date.select("date_sk","full_date"),
        invoice_keyed["date_sk"] == dim_date["date_sk"], "left"
    ) \
    .join(
        customer_sv.select("CustomerId","SupportRepId"),
        "CustomerId", "left"
    ) \
    .join(
        dim_employee.select("employee_nk","employee_key"),
        customer_sv["SupportRepId"] == dim_employee["employee_nk"], "left"
    ) \
    .select(
        invoiceline["InvoiceLineId"].alias("invoice_line_id"),
        invoice_keyed["InvoiceId"].alias("invoice_id"),
        col("customer_key"),
        col("track_key"),
        invoice_keyed["date_sk"],
        col("employee_key"),
        invoice_keyed["BillingCountry"].alias("billing_country"),
        invoice_keyed["BillingState"].alias("billing_state"),
        invoiceline["Quantity"].alias("quantity"),
        invoiceline["UnitPrice"].cast("decimal(10,2)").alias("unit_price"),
        (invoiceline["Quantity"] * invoiceline["UnitPrice"]).cast("decimal(10,2)").alias("line_total"),
        invoice_keyed["Total"].cast("decimal(10,2)").alias("invoice_total")
    )

sales_fact.write.format("delta").mode("overwrite") \
    .option("overwriteSchema","true") \
    .saveAsTable(f"{GOLD}.fact_sales")
print(f"✅ FACT_SALES: {sales_fact.count()} rows")

In [0]:
# ── FACT_SALES ────────────────────────────────────────────────────────────────
#
# Grain: one invoice line item.
#
# dim_customer join uses a date-range condition so each fact row links to
# the customer's SCD2 version that was current at invoice time, not just
# the latest version. This is what makes SCD Type 2 meaningful in the fact.
#
# dim_employee is also SCD2 — same date-range join applied.
# ─────────────────────────────────────────────────────────────────────────────
print("Building FACT_SALES...")

invoice     = spark.table(f"{SILVER}.invoice")
invoiceline = spark.table(f"{SILVER}.invoiceline")
customer_sv = spark.table(f"{SILVER}.customer")

dim_customer_hist = spark.table(f"{GOLD}.dim_customer")
dim_employee_hist = spark.table(f"{GOLD}.dim_employee")
dim_track         = spark.table(f"{GOLD}.dim_track")
dim_date          = spark.table(f"{GOLD}.dim_date")

# Cast InvoiceDate to timestamp for the date-range join
invoice_keyed = invoice \
    .withColumn("date_sk",      date_format(col("InvoiceDate"), "yyyyMMdd").cast("int")) \
    .withColumn("InvoiceDateTs", col("InvoiceDate").cast("timestamp"))

# Bring SupportRepId onto each invoice via the silver customer table
invoice_with_rep = invoice_keyed.join(
    customer_sv.select("CustomerId", "SupportRepId"),
    on="CustomerId",
    how="left"
)

sales_fact = invoiceline \
    .join(invoice_with_rep, "InvoiceId", "inner") \
    .join(
        # SCD2 date-range join: invoice date must fall within the customer version's active window
        dim_customer_hist.select(
            col("customer_id"), col("customer_key"),
            col("effective_start_date").alias("cust_start"),
            col("effective_end_date").alias("cust_end")
        ),
        (
            invoice_with_rep["CustomerId"] == dim_customer_hist["customer_id"]
        ) & (
            col("InvoiceDateTs") >= col("cust_start")
        ) & (
            col("cust_end").isNull() | (col("InvoiceDateTs") < col("cust_end"))
        ),
        "left"
    ) \
    .join(
        dim_track.select("track_nk", "track_key"),
        invoiceline["TrackId"] == dim_track["track_nk"],
        "left"
    ) \
    .join(
        dim_date.select("date_sk", "full_date"),
        invoice_with_rep["date_sk"] == dim_date["date_sk"],
        "left"
    ) \
    .join(
        # SCD2 date-range join for employee as well
        dim_employee_hist.select(
            col("employee_nk"), col("employee_key"),
            col("effective_start_date").alias("emp_start"),
            col("effective_end_date").alias("emp_end")
        ),
        (
            invoice_with_rep["SupportRepId"] == dim_employee_hist["employee_nk"]
        ) & (
            col("InvoiceDateTs") >= col("emp_start")
        ) & (
            col("emp_end").isNull() | (col("InvoiceDateTs") < col("emp_end"))
        ),
        "left"
    ) \
    .select(
        invoiceline["InvoiceLineId"].alias("invoice_line_id"),
        invoice_with_rep["InvoiceId"].alias("invoice_id"),
        col("customer_key"),
        col("track_key"),
        invoice_with_rep["date_sk"],
        col("employee_key"),
        invoice_with_rep["BillingCountry"].alias("billing_country"),
        invoice_with_rep["BillingState"].alias("billing_state"),
        invoiceline["Quantity"].alias("quantity"),
        invoiceline["UnitPrice"].cast("decimal(10,2)").alias("unit_price"),
        (invoiceline["Quantity"] * invoiceline["UnitPrice"]).cast("decimal(10,2)").alias("line_total"),
        invoice_with_rep["Total"].cast("decimal(10,2)").alias("invoice_total")
    )

sales_fact.write.format("delta").mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable(f"{GOLD}.fact_sales")
print(f"✅ FACT_SALES: {sales_fact.count()} rows")

In [0]:
# ── FACT_SALES_CUSTOMER_AGG ───────────────────────────────────────────────────
# Pre-aggregated rollup per customer — uses is_current=true for readable names.
# ─────────────────────────────────────────────────────────────────────────────
print("Building FACT_SALES_CUSTOMER_AGG...")
fact_sales   = spark.table(f"{GOLD}.fact_sales")
dim_customer = spark.table(f"{GOLD}.dim_customer").filter("is_current = true")

agg_fact = fact_sales \
    .join(
        dim_customer.select("customer_key", "customer_name", "country", "city"),
        "customer_key",
        "left"
    ) \
    .groupBy("customer_key", "customer_name", "country", "city") \
    .agg(
        count("invoice_line_id").alias("total_line_items"),
        sum("line_total").cast("decimal(12,2)").alias("total_revenue"),
        avg("line_total").cast("decimal(10,2)").alias("avg_order_value"),
        count("invoice_id").alias("total_invoices"),
        max("line_total").cast("decimal(10,2)").alias("max_order_value"),
        min("line_total").cast("decimal(10,2)").alias("min_order_value")
    )

agg_fact.write.format("delta").mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable(f"{GOLD}.fact_sales_customer_agg")
print(f"✅ FACT_SALES_CUSTOMER_AGG: {agg_fact.count()} rows")